In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
# Ruta de la carpeta que contiene los archivos de desglosados zonal
bitacora = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Reporte_fin_semana/bitacora_compilada.xlsx')

bitacora.head()

,Patio,Fecha,CÃ³digo conductor,Conductor,Servicio,Hora Inicio,Hora Fin,Origen,Destino,Ruta,Tarea,Distancia Km,Tabla,Tipo de Vehiculo,Km. Perdidos,Vehiculo,Acciones
0,Tintal1,20260530,507924 510422,JULIAN DAVID FERNANDEZ ARDILA ...,CE1220001,03:15:00,06:21:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",1,BUSETON,NaN,Z50-4184,NaN
1,Tintal1,20260530,Cambio de turno operativo ...,DANIEL ALEJANDRO IMBACHI MARTINEZ ...,CE1220002,03:25:00,06:30:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",2,BUSETON,NaN,Z50-4418,NaN
2,Tintal1,20260530,510497,JORGE DAVID AVENDAÃ‘O BARRAGAN,CE11F0001,03:30:00,04:44:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,614_IDA_FMS,"27,887",3,BUSETON,NaN,Z50-4401,NaN
3,Tintal1,20260530,509671,CRISTIAN DAVID PARRA COMBITA,CE1220003,03:35:00,06:42:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",3,BUSETON,NaN,Z50-4140,NaN
4,Tintal1,20260530,509961,BRANNY MOHAMED LEGUIZAMO VALENCIANO,CE11F0002,03:41:00,04:55:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,614_IDA_FMS,"27,887",5,BUSETON,NaN,Z50-4133,NaN


In [3]:
bitacora["estado_op"] = (
    bitacora["Conductor"]
    .str.contains("NO ASIGNADO", case=False, na=False)
    .astype(int)
)

bitacora.head()

,Patio,Fecha,CÃ³digo conductor,Conductor,Servicio,Hora Inicio,Hora Fin,Origen,Destino,Ruta,Tarea,Distancia Km,Tabla,Tipo de Vehiculo,Km. Perdidos,Vehiculo,Acciones,estado_op
0,Tintal1,20260530,507924 510422,JULIAN DAVID FERNANDEZ ARDILA ...,CE1220001,03:15:00,06:21:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",1,BUSETON,NaN,Z50-4184,NaN,0
1,Tintal1,20260530,Cambio de turno operativo ...,DANIEL ALEJANDRO IMBACHI MARTINEZ ...,CE1220002,03:25:00,06:30:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",2,BUSETON,NaN,Z50-4418,NaN,0
2,Tintal1,20260530,510497,JORGE DAVID AVENDAÃ‘O BARRAGAN,CE11F0001,03:30:00,04:44:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,614_IDA_FMS,"27,887",3,BUSETON,NaN,Z50-4401,NaN,0
3,Tintal1,20260530,509671,CRISTIAN DAVID PARRA COMBITA,CE1220003,03:35:00,06:42:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,DL219_FMS_2,"59,097",3,BUSETON,NaN,Z50-4140,NaN,0
4,Tintal1,20260530,509961,BRANNY MOHAMED LEGUIZAMO VALENCIANO,CE11F0002,03:41:00,04:55:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,614_IDA_FMS,"27,887",5,BUSETON,NaN,Z50-4133,NaN,0


In [4]:
# Función para separar el texto y los códigos
def separar_novedad(texto):
    if pd.isna(texto):
        return pd.Series([None, None, None])

    # Limpiar espacios
    texto = texto.strip()

    # Buscar todos los números en la cadena
    numeros = re.findall(r'\d+', texto)

    # Extraer el texto antes del primer número (estado_novedad)
    estado = re.split(r'\d+', texto, maxsplit=1)[0].strip()

    # Retornar con estructura: [estado, operador_cambio, operador_faltante]
    operador_cambio = numeros[0] if len(numeros) > 0 else None
    operador_faltante = numeros[1] if len(numeros) > 1 else None

    return pd.Series([estado, operador_cambio, operador_faltante])

# Aplicar función
bitacora[['estado_novedad', 'operador_cambio', 'operador_faltante']] = bitacora['CÃ³digo conductor'].apply(separar_novedad)

bitacora.head()

,Patio,Fecha,CÃ³digo conductor,Conductor,Servicio,Hora Inicio,Hora Fin,Origen,Destino,Ruta,...,Distancia Km,Tabla,Tipo de Vehiculo,Km. Perdidos,Vehiculo,Acciones,estado_op,estado_novedad,operador_cambio,operador_faltante
0,Tintal1,20260530,507924 510422,JULIAN DAVID FERNANDEZ ARDILA ...,CE1220001,03:15:00,06:21:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,...,"59,097",1,BUSETON,NaN,Z50-4184,NaN,0,,507924,510422
1,Tintal1,20260530,Cambio de turno operativo ...,DANIEL ALEJANDRO IMBACHI MARTINEZ ...,CE1220002,03:25:00,06:30:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,...,"59,097",2,BUSETON,NaN,Z50-4418,NaN,0,Cambio de turno operativo,508430,509657
2,Tintal1,20260530,510497,JORGE DAVID AVENDAÃ‘O BARRAGAN,CE11F0001,03:30:00,04:44:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,...,"27,887",3,BUSETON,NaN,Z50-4401,NaN,0,,510497,None
3,Tintal1,20260530,509671,CRISTIAN DAVID PARRA COMBITA,CE1220003,03:35:00,06:42:00,072A05_TM_(Engativa),072A05_TM_(Engativa),DL219,...,"59,097",3,BUSETON,NaN,Z50-4140,NaN,0,,509671,None
4,Tintal1,20260530,509961,BRANNY MOHAMED LEGUIZAMO VALENCIANO,CE11F0002,03:41:00,04:55:00,206A06_TM_(Montevideo),525A12_TM_( BOLONIA ),614,...,"27,887",5,BUSETON,NaN,Z50-4133,NaN,0,,509961,None


In [5]:
# Ordenar los datos como corresponde
bitacora = bitacora.sort_values(
    by=['Conductor', 'Servicio', 'Hora Inicio', 'Ruta'],
    ascending=[True, True, True, True]
).reset_index(drop=True)

# Crear la columna 'parte' numerando consecutivamente
# cada grupo de (Conductor, Servicio)
bitacora['parte'] = (
    bitacora.groupby(['Conductor', 'Servicio'])
    .cumcount() + 1
)

bitacora.head()

,Patio,Fecha,CÃ³digo conductor,Conductor,Servicio,Hora Inicio,Hora Fin,Origen,Destino,Ruta,...,Tabla,Tipo de Vehiculo,Km. Perdidos,Vehiculo,Acciones,estado_op,estado_novedad,operador_cambio,operador_faltante,parte
0,Tintal2,20260530,507680,ADAN ALBERTO ALVARADO PEREZ,CE16A0004,15:17:30,18:51:30,072A05_TM_(Engativa),565A12_TM_(SOCHES),539,...,5,PADRON,NaN,Z50-7037,NaN,0,,507680,None,1
1,Tintal2,20260530,507680,ADAN ALBERTO ALVARADO PEREZ,CE16A0016,20:31:00,22:34:00,565A12_TM_(SOCHES),072A05_TM_(Engativa),539,...,17,PADRON,NaN,Z50-7134,NaN,0,,507680,None,1
2,Troncal,20260531,500826,AGUIRRE TORRES DANIEL ENRIQUE,GMBI260524005,17:06:00,18:11:00,Portal Usme T2A,Portal Eldorado T-4,H54 - K54,...,10,BIARTICULADO,NaN,E074,NaN,0,,500826,None,1
3,Troncal,20260531,500826,AGUIRRE TORRES DANIEL ENRIQUE,GMBI260524005,18:16:00,19:15:00,Portal Eldorado T-4,Portal Usme T2A,H54 - K54,...,10,BIARTICULADO,NaN,E074,NaN,0,,500826,None,2
4,Troncal,20260531,500826,AGUIRRE TORRES DANIEL ENRIQUE,GMBI260524005,19:19:00,20:21:30,Portal Usme T2A,Portal Eldorado T-4,H54 - K54,...,10,BIARTICULADO,NaN,E074,NaN,0,,500826,None,3


In [6]:
conteo_bitacora = (
    bitacora
    .groupby(["Fecha", "Ruta", "estado_novedad"])
    .size()
    .reset_index(name="cantidad")
    .sort_values(["Fecha", "Ruta", "cantidad"], ascending=[True, True, False])
)

conteo_bitacora

,Fecha,Ruta,estado_novedad,cantidad
0,20260530,1-1 ALAMOS,,78
1,20260530,1-1 ALAMOS,Ausentismo,6
2,20260530,1-9 VILLAS DEL DORADO,,56
3,20260530,1-9 VILLAS DEL DORADO,Ausentismo,16
4,20260530,12,,43
...,...,...,...,...
240,20260531,SE10,,39
241,20260531,SE10,Ausentismo,8
242,20260531,SE14,,54
243,20260531,SE14,Ausentismo,2


In [7]:
bitacora_filtrada = bitacora[
    bitacora["estado_novedad"].notna() & 
    (bitacora["estado_novedad"].str.strip() != "")
]

In [8]:
conteo_bitacora = (
    bitacora_filtrada
    .groupby(["Fecha", "Ruta", "estado_novedad"])
    .size()
    .reset_index(name="cantidad")
    .sort_values(["Fecha", "Ruta", "cantidad"], ascending=[True, True, False])
)

conteo_bitacora

,Fecha,Ruta,estado_novedad,cantidad
0,20260530,1-1 ALAMOS,Ausentismo,6
1,20260530,1-9 VILLAS DEL DORADO,Ausentismo,16
2,20260530,12,Ausentismo,2
3,20260530,12,Cambio de turno,1
4,20260530,142,Ausentismo,5
...,...,...,...,...
138,20260531,Ruta fÃ¡cil 5,Cambio de turno,3
139,20260531,Ruta fÃ¡cil 6,Cambio de turno,3
140,20260531,SE10,Ausentismo,8
141,20260531,SE14,Ausentismo,2


In [9]:
bitacora_ausentismo = bitacora[
    bitacora["estado_novedad"].notna() &
    (bitacora["estado_novedad"].str.strip() == "Ausentismo")
]

In [10]:
top_ruta_ausentismo = (
    bitacora_ausentismo
    .groupby("Ruta")
    .size()
    .reset_index(name="cantidad_ausentismos")
    .sort_values("cantidad_ausentismos", ascending=False)
)

top_ruta_ausentismo.head()

,Ruta,cantidad_ausentismos
25,DD204,43
14,614,28
12,576,26
20,BD237,24
13,577,18


In [11]:
pivot_ausentismo_dia = (
    bitacora_ausentismo
    .groupby(["Fecha", "Ruta"])
    .size()
    .unstack(fill_value=0)
)

pivot_ausentismo_dia

Ruta,1-1 ALAMOS,1-9 VILLAS DEL DORADO,12,142,16-3 ALAMOS,16-5 VILLA AMALIA,16-6 LA FAENA,359,402,466,...,DH216,DL219,E25,F23-J23,H54 - K54,KB309,KL307,L10 - K10,SE10,SE14
Fecha,,,,,,,,,,,,,,,,,,,,,
20260530,6,16,2,5,0,0,0,0,5,2,...,0,2,5,4,4,3,2,1,8,6
20260531,0,0,0,8,7,14,4,3,3,11,...,6,9,9,0,0,2,2,0,8,2


In [12]:
ausentismo_por_dia = (
    bitacora_ausentismo
    .groupby("Fecha")
    .size()
    .reset_index(name="total_ausentismos")
    .sort_values("Fecha")
)

ausentismo_por_dia

,Fecha,total_ausentismos
0,20260530,197
1,20260531,212


In [13]:
bitacora_ausentismo = bitacora[
    bitacora["estado_novedad"].notna() &
    (bitacora["estado_novedad"].str.strip() == "Ausentismo")
]

In [14]:
top_patio_ausentismo = (
    bitacora_ausentismo
    .groupby("Patio")
    .size()
    .reset_index(name="cantidad_ausentismos")
    .sort_values("cantidad_ausentismos", ascending=False)
)

top_patio_ausentismo

,Patio,cantidad_ausentismos
4,Verbena,141
1,Tintal1,134
2,Tintal2,103
3,Troncal,17
0,La Y,14


In [15]:
top_patio_ausentismo = (
    bitacora_ausentismo
    .groupby("Patio")["Servicio"]
    .nunique()
    .reset_index(name="servicios_unicos_ausentismo")
    .sort_values("servicios_unicos_ausentismo", ascending=False)
)

top_patio_ausentismo

,Patio,servicios_unicos_ausentismo
4,Verbena,78
1,Tintal1,60
2,Tintal2,37
0,La Y,12
3,Troncal,6


In [16]:
top_ruta_ausentismo_dia = (
    bitacora_ausentismo
    .groupby(["Fecha", "Ruta"])
    .size()
    .reset_index(name="cantidad_ausentismos")
    .sort_values(["Fecha", "cantidad_ausentismos"], ascending=[True, False])
)

top_ruta_ausentismo_dia.head()

,Fecha,Ruta,cantidad_ausentismos
7,20260530,576,20
20,20260530,DD204,19
1,20260530,1-9 VILLAS DEL DORADO,16
9,20260530,614,16
15,20260530,BD237,11


In [17]:
top5_ruta_por_dia = (
    top_ruta_ausentismo_dia
    .groupby("Fecha")
    .head(5)
)

top5_ruta_por_dia

,Fecha,Ruta,cantidad_ausentismos
7,20260530,576,20
20,20260530,DD204,19
1,20260530,1-9 VILLAS DEL DORADO,16
9,20260530,614,16
15,20260530,BD237,11
51,20260531,DD204,24
34,20260531,16-5 VILLA AMALIA,14
39,20260531,5-4 FLORIDA,13
46,20260531,BD237,13
43,20260531,614,12


In [18]:
top_ruta_ausentismo_dia = (
    bitacora_ausentismo
    .groupby(["Fecha", "Ruta"])["Servicio"]
    .nunique()
    .reset_index(name="servicios_unicos_ausentismo")
    .sort_values(["Fecha", "servicios_unicos_ausentismo"], ascending=[True, False])
)

top_ruta_ausentismo_dia.head()

,Fecha,Ruta,servicios_unicos_ausentismo
15,20260530,BD237,8
7,20260530,576,7
6,20260530,539,6
4,20260530,402,5
9,20260530,614,5


In [19]:
top5_ruta_por_dia = (
    top_ruta_ausentismo_dia
    .groupby("Fecha")
    .head(5)
)

top5_ruta_por_dia

,Fecha,Ruta,servicios_unicos_ausentismo
15,20260530,BD237,8
7,20260530,576,7
6,20260530,539,6
4,20260530,402,5
9,20260530,614,5
46,20260531,BD237,8
44,20260531,740,7
55,20260531,DL219,7
38,20260531,466,6
43,20260531,614,6


In [20]:
pivot_ausentismo_patio = (
    bitacora_ausentismo
    .groupby(["Fecha", "Patio"])["Servicio"]
    .nunique()
    .unstack(fill_value=0)
)

pivot_ausentismo_patio

Patio,La Y,Tintal1,Tintal2,Troncal,Verbena
Fecha,,,,,
20260530,6,25,20,6,38
20260531,6,35,17,0,40


In [21]:
import pandas as pd

data = {
    "Patio": ["Verbena", "La Y", "Tintal 1", "Tintal 2", "Troncal"],
    "Pendientes por cubrir": [27, 0, 20,12 , 0]
}

df_pendientes = pd.DataFrame(data)

df_pendientes

,Patio,Pendientes por cubrir
0,Verbena,27
1,La Y,0
2,Tintal 1,20
3,Tintal 2,12
4,Troncal,0


In [35]:
import pandas as pd

# Crear estructura de columnas multinivel
columns = pd.MultiIndex.from_tuples([
    ("TM01", "Sábado"),
    ("TM01", "Domingo"),
    ("TM02", "Sábado"),
    ("TM02", "Domingo"),
])

data = [
    [4, 1, 2, 0],   # Verbena
    [0, 0, 0, 0],   # La Y
    [1, 0, 0, 0],   # Tintal 1
    [0, 0, 0, 0],   # Tintal 2
    [1, 0, 0, 0],   # Troncal
]

index = ["Verbena", "La Y", "Tintal 1", "Tintal 2", "Troncal"]

df_tm = pd.DataFrame(data, index=index, columns=columns)

df_tm

TM01           TM02        
         Sábado Domingo Sábado Domingo
Verbena       4       1      2       0
La Y          0       0      0       0
Tintal 1      1       0      0       0
Tintal 2      0       0      0       0
Troncal       1       0      0       0